# SENSEI — Session Intelligence
## Module 1: Purchase Prediction · Exploratory Data Analysis

> SENSEI turns raw e-commerce clickstream data into session-level intelligence.  
> This notebook explores the raw RetailRocket event log to understand visitor behaviour  
> before building the SENSEI session feature store.

**Dataset:** [RetailRocket E-Commerce Dataset](https://www.kaggle.com/datasets/retailrocket/ecommerce-dataset)

## Contents
1. Load & inspect data
2. Event distribution
3. Bot detection & removal
4. Visitor behaviour & conversion funnel
5. Time patterns
6. Key findings & feature motivation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

sys.path.append(os.path.join('..', 'src'))

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

## 1. Load & Inspect

In [ ]:
DATA_DIR = os.path.join('..', 'data')

df = pd.read_csv(os.path.join(DATA_DIR, 'events.csv'))
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')

print(f'Shape before deduplication: {df.shape}')
n_dupes = df.duplicated().sum()
print(f'Duplicate rows            : {n_dupes:,}')
df = df.drop_duplicates().reset_index(drop=True)
print(f'Shape after deduplication : {df.shape}')
df.head()

In [ ]:
# Bot / outlier detection
# Visitors with an extreme number of events are likely crawlers or test accounts.
events_per_visitor = df.groupby('visitorid').size()

threshold = 500
n_bots = (events_per_visitor > threshold).sum()
pct_bots = n_bots / len(events_per_visitor) * 100

print(f'Visitors with > {threshold} events : {n_bots:,} ({pct_bots:.2f} % of visitors)')
print(f'Events from these visitors         : {events_per_visitor[events_per_visitor > threshold].sum():,}')

fig, ax = plt.subplots(figsize=(10, 3))
sns.histplot(events_per_visitor.clip(upper=200), bins=100, ax=ax, color='#4C72B0')
ax.axvline(threshold, color='red', linestyle='--', label=f'Bot threshold ({threshold})')
ax.set_xlabel('Events per visitor (capped at 200)')
ax.set_ylabel('Visitors')
ax.set_title('Events per visitor — identifying potential bots')
ax.legend()
plt.tight_layout()
plt.show()

# Remove bot visitors from the dataframe
bot_visitors = events_per_visitor[events_per_visitor > threshold].index
df = df[~df['visitorid'].isin(bot_visitors)].reset_index(drop=True)
print(f'\nShape after bot removal: {df.shape}')

In [ ]:
df.info()

In [ ]:
print('=== Missing values ===')
print(df.isnull().sum())
print()
print('Note: transactionid is only set for transaction events — NaN for views/addtocart is expected.')

In [ ]:
print(f"Date range: {df['timestamp'].min()} → {df['timestamp'].max()}")
print(f"Unique visitors : {df['visitorid'].nunique():,}")
print(f"Unique items    : {df['itemid'].nunique():,}")

event_counts = df['event'].value_counts().reset_index()
event_counts.columns = ['event', 'count']
event_counts['pct (%)'] = (event_counts['count'] / len(df) * 100).round(2)

print(event_counts.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=event_counts, x='event', y='count', ax=ax, palette='muted')
ax.set_title('Event counts')
ax.set_ylabel('Number of events')
ax.set_xlabel('')
for i, row in event_counts.iterrows():
    ax.text(i, row['count'] * 1.01,
            f"{row['count']:,}\n({row['pct (%)']:.1f}%)",
            ha='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
event_counts = df['event'].value_counts().reset_index()
event_counts.columns = ['event', 'count']
event_counts['pct (%)'] = (event_counts['count'] / len(df) * 100).round(2)

print(event_counts.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.barplot(data=event_counts, x='event', y='count', ax=axes[0], palette='muted')
axes[0].set_title('Event counts')
axes[0].set_ylabel('Number of events')
axes[0].set_xlabel('')
for i, row in event_counts.iterrows():
    axes[0].text(i, row['count'] * 1.01, f"{row['count']:,}", ha='center', fontsize=9)

sns.barplot(data=event_counts, x='event', y='pct (%)', ax=axes[1], palette='muted')
axes[1].set_title('Event share (%)')
axes[1].set_ylabel('Share (%)')
axes[1].set_xlabel('')
for i, row in event_counts.iterrows():
    axes[1].text(i, row['pct (%)'] * 1.01, f"{row['pct (%)']:.1f}%", ha='center', fontsize=9)

plt.tight_layout()
plt.show()

**Finding:** The funnel is extremely skewed — 96.7 % views, 2.5 % add-to-cart, 0.8 % transactions. This class imbalance will be the main challenge for modelling.

## 3. Visitor Behaviour

In [ ]:
events_per_visitor = df.groupby('visitorid').size()

print('Events per visitor:')
print(events_per_visitor.describe().round(1))

In [ ]:
# Visitors who purchased vs. never purchased
buyers = df[df['event'] == 'transaction']['visitorid'].unique()
print(f'Visitors who purchased at least once : {len(buyers):,}')
print(f'Visitors who never purchased          : {df["visitorid"].nunique() - len(buyers):,}')
print(f'Purchase conversion rate (visitors)   : {len(buyers) / df["visitorid"].nunique() * 100:.2f} %')

In [ ]:
# Conversion funnel — at visitor level
# How many visitors reached each stage of the funnel?
all_visitors   = set(df['visitorid'].unique())
viewed         = set(df[df['event'] == 'view']['visitorid'].unique())
added_to_cart  = set(df[df['event'] == 'addtocart']['visitorid'].unique())
transacted     = set(df[df['event'] == 'transaction']['visitorid'].unique())

funnel = pd.DataFrame({
    'Stage': ['Viewed', 'Added to cart', 'Purchased'],
    'Visitors': [len(viewed), len(added_to_cart), len(transacted)],
})
funnel['% of total visitors'] = (funnel['Visitors'] / len(all_visitors) * 100).round(2)
funnel['% of previous stage'] = [
    100.0,
    len(added_to_cart) / len(viewed) * 100,
    len(transacted) / len(added_to_cart) * 100,
]
funnel['% of previous stage'] = funnel['% of previous stage'].round(2)

print(funnel.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=funnel, x='Stage', y='Visitors', ax=ax, palette='muted')
ax.set_title('Visitor conversion funnel')
ax.set_ylabel('Unique visitors')
ax.set_xlabel('')
for i, row in funnel.iterrows():
    ax.text(i, row['Visitors'] * 1.01,
            f"{row['Visitors']:,}\n({row['% of total visitors']:.1f}%)",
            ha='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
df['hour']    = df['timestamp'].dt.hour
df['weekday'] = df['timestamp'].dt.day_name()
df['week']    = df['timestamp'].dt.isocalendar().week.astype(int)

WEEKDAY_ORDER = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

hourly = df.groupby('hour').size().reset_index(name='count')
daily  = (
    df.groupby('weekday').size()
    .reindex(WEEKDAY_ORDER)
    .reset_index(name='count')
)
daily['weekday_short'] = daily['weekday'].str[:3]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.lineplot(data=hourly, x='hour', y='count', marker='o', ax=axes[0], color='#4C72B0')
axes[0].set_title('Events by hour of day (UTC)\nNote: RetailRocket is based in Russia — UTC+3 likely applies')
axes[0].set_xlabel('Hour (UTC)')
axes[0].set_ylabel('Events')
axes[0].set_xticks(range(0, 24, 2))

sns.barplot(data=daily, x='weekday_short', y='count', ax=axes[1], palette='muted',
            order=[d[:3] for d in WEEKDAY_ORDER])
axes[1].set_title('Events by weekday')
axes[1].set_xlabel('')
axes[1].set_ylabel('Events')

plt.tight_layout()
plt.show()

## 5. Key Findings

| # | Finding |
|---|---|
| 1 | Raw event log has 2,756,101 rows from 1,407,580 unique visitors (May–Sep 2015) |
| 2 | After deduplication and bot removal (~visitors with >500 events), the dataset is cleaner |
| 3 | Event funnel: **~97 % view → ~2.5 % add-to-cart → ~0.8 % transaction** |
| 4 | Only ~1 % of visitors ever complete a purchase — strong class imbalance |
| 5 | The add-to-cart → purchase step loses ~68 % of visitors — biggest drop in the funnel |
| 6 | Peak activity appears in midday hours (UTC); actual local time is UTC+3 for this dataset |
| 7 | `transactionid` is only non-null for transaction events — no unexpected data quality issues |
| 8 | A single `transactionid` can cover multiple items — one transaction ≠ one event row |

**Motivated features for modelling:**
- `n_views`, `n_addtocart` → engagement depth
- `n_items` → breadth of interest
- `n_revisited_items` → strong re-engagement / intent signal
- `view_to_cart_ratio` → decisiveness
- `duration_sec` → session engagement length
- `hour_of_day`, `day_of_week` → temporal purchase patterns
- `is_first_session` → returning visitors behave differently

**Next step → notebook 02: Sessionise and build tabular features**

In [ ]:
df['hour']    = df['timestamp'].dt.hour
df['weekday'] = df['timestamp'].dt.day_name()
df['week']    = df['timestamp'].dt.isocalendar().week.astype(int)

WEEKDAY_ORDER = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

hourly = df.groupby('hour').size().reset_index(name='count')
daily  = (
    df.groupby('weekday').size()
    .reindex(WEEKDAY_ORDER)
    .reset_index(name='count')
)
daily['weekday_short'] = daily['weekday'].str[:3]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.lineplot(data=hourly, x='hour', y='count', marker='o', ax=axes[0], color='#4C72B0')
axes[0].set_title('Events by hour of day')
axes[0].set_xlabel('Hour (UTC)')
axes[0].set_ylabel('Events')
axes[0].set_xticks(range(0, 24, 2))

sns.barplot(data=daily, x='weekday_short', y='count', ax=axes[1], palette='muted',
            order=[d[:3] for d in WEEKDAY_ORDER])
axes[1].set_title('Events by weekday')
axes[1].set_xlabel('')
axes[1].set_ylabel('Events')

plt.tight_layout()
plt.show()

## 5. Key Findings

| # | Finding |
|---|---|
| 1 | **2,756,101 events** from **1,407,580 unique visitors** over ~4.5 months (May–Sep 2015) |
| 2 | Event funnel: **96.7 % view → 2.5 % add-to-cart → 0.8 % transaction** |
| 3 | Most visitors are one-time or low-engagement (highly right-skewed events-per-visitor distribution) |
| 4 | Only **~1 %** of visitors ever complete a purchase |
| 5 | Peak activity: **midday hours**, weekday pattern shows weekday-heaviness |
| 6 | `transactionid` is only non-null for transaction events — no data quality issues otherwise |

**Next step → notebook 02: Sessionize and build tabular features**